# 🚀 Huấn Luyện & Đánh Giá Toàn Bộ 7 Cấu Hình Ablation COCOA Trên NVIDIA A100 (40GB/80GB VRAM)

**Dự án:** Khóa luận Tốt nghiệp — Dự đoán biên dạng hoàn chỉnh của vật thể bị che khuất (**Amodal Shape Prediction**)  
**Tác giả:** Võ Thế Nguyễn (Ryan) & Phan Quang Nghĩa  
**Môi trường:** Google Colab Pro / Pro+ với GPU **NVIDIA A100**  

---

### 📌 Bối Cảnh Kỹ Thuật & Cập Nhật Quan Trọng (Fix Category ID COCOA):
Trong commit mới nhất (`a4e5b37b`), hệ thống đã sửa lỗi nghiêm trọng: **Trích xuất Category ID đa tầng cho tập COCOA** (`extract_cocoa_category_id`).
- **Lỗi cũ:** Code cũ gọi `ann.get('category_id', 0)`. Do cấu trúc COCOA lưu nhãn bên trong từng `region`, lệnh này luôn trả về `0` (100% các mẫu rơi vào class 0).
- **Phân tích tác động lên 7 cấu hình:**
  1. **Row 1, Row 2, Row 3, Row 5:** **KHÔNG CÓ** khối Category Embedding (forward pass chỉ nhận `inputs`, không dùng `class_ids`). Trọng số và kết quả cũ của 4 hàng này **hoàn toàn chính xác, không bị ảnh hưởng** bởi bug.
  2. **Row 4 (Main Config), Row 6 (Full Config), Row 7 (Proposed SG-GBA):** **BẮT BUỘC TRAIN LẠI**. Vì 3 cấu hình này có khối `Category Embedding (91 lớp)`. Lần train cũ chỉ có class 0 được học, 90 class còn lại không có đặc trưng ngữ nghĩa. Với bản sửa lỗi này, 3 hàng này sẽ học trọn vẹn biểu diễn ngữ nghĩa của cả 91 lớp đối tượng.

---

### ⚡ Các Tính Năng Nổi Bật Của Notebook Này:
1. **Cơ Chế Cách Ly Tuyệt Đối (Zero Overwrite Isolation):**
   - Checkpoints lưu tại: `/content/drive/MyDrive/KLTN_Amodal_2026/checkpoints/cocoa_ablation_v2/`
   - Results JSON lưu tại: `/content/drive/MyDrive/KLTN_Amodal_2026/results/cocoa_ablation_v2/`
   - **Hoàn toàn không bao giờ ghi đè** lên các checkpoint cũ ở `checkpoints/cocoa/` hay `swin_amodal_epoch_*.pth`.
2. **Linh Hoạt 2 Chiến Lược Huấn Luyện:**
   - **Chiến lược 1 (Khuyên dùng - Tiết kiệm thời gian ~1.5h):** Chỉ chạy 3 cấu hình bị ảnh hưởng (**Row 4, Row 6, Row 7**).
   - **Chiến lược 2 (Chuẩn mực học thuật cao nhất ~3h):** Chạy mới toanh toàn bộ **7 cấu hình (Row 1 -> Row 7)** từ đầu để tạo bảng Table V đồng nhất 100%.
   - **Chạy lẻ từng cấu hình:** Cung cấp cell riêng biệt cho từng Row nếu muốn chạy ngắt quãng.
3. **Tối Ưu Hóa Tối Đa Trên GPU A100:**
   - Batch Size 16, Mixed Precision (AMP FP16), TF32 Tensor Cores, `num_workers=8`.
   - Tự động nạp dữ liệu thông minh (quét Drive trước, nếu chưa có thì tải trực tiếp từ server vào NVMe SSD của Colab).

### ⚡ Bước 1: Kiểm Tra GPU A100 & Thông Số Phần Cứng Colab
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 GPU -> High-RAM**.

In [ ]:
!nvidia-smi

import os
import psutil
import torch

print("\n" + "=" * 65)
print("THÔNG TIN HỆ THỐNG & PHẦN CỨNG COLAB")
print("=" * 65)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    compute_cap = torch.cuda.get_device_capability(0)
    print(f"GPU Name        : {gpu_name}")
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print(f"Compute Cap     : {compute_cap[0]}.{compute_cap[1]}")

    # Kích hoạt tăng tốc phần cứng cho NVIDIA Ampere A100
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
        print("⚡ Đã kích hoạt Tensor Cores TF32 (float32_matmul_precision = 'high')")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Vui lòng vào Runtime -> Change runtime type -> A100 GPU")

cpu_count = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"vCPUs Available : {cpu_count} cores (Tối ưu cho num_workers=8)")
print(f"System RAM      : {ram_gb:.2f} GB")
print("=" * 65)

### 📁 Bước 2: Kết Nối Google Drive & Thiết Lập Thư Mục Lưu Trữ Độc Lập (`cocoa_ablation_v2`)
Mọi file checkpoint `.pth` và kết quả đánh giá `.json` sẽ được lưu thẳng vào thư mục riêng `cocoa_ablation_v2` trên Google Drive để không bị ghi đè lên các checkpoint cũ.

In [ ]:
from google.colab import drive
import os
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Thiết lập thư mục gốc KLTN trên Google Drive
DRIVE_DIR = "/content/drive/MyDrive/KLTN_Amodal_2026"
if not os.path.exists(DRIVE_DIR) and os.path.exists("/content/drive/MyDrive/amodal_shape_prediction"):
    DRIVE_DIR = "/content/drive/MyDrive/amodal_shape_prediction"

# 3. Thiết lập TAG thư mục mới cách ly 100% để chống đè dữ liệu cũ
EXP_TAG = "cocoa_ablation_v2"
CKPT_SAVE_DIR = f"{DRIVE_DIR}/checkpoints/{EXP_TAG}"
RESULTS_SAVE_DIR = f"{DRIVE_DIR}/results/{EXP_TAG}"
LOGS_SAVE_DIR = f"{DRIVE_DIR}/logs/{EXP_TAG}"

os.makedirs(CKPT_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_SAVE_DIR, exist_ok=True)
os.makedirs(LOGS_SAVE_DIR, exist_ok=True)

print(f"✅ Thư mục Google Drive đã sẵn sàng:")
print(f"   • Checkpoints : {CKPT_SAVE_DIR}")
print(f"   • Results     : {RESULTS_SAVE_DIR}")
print(f"   • Logs        : {LOGS_SAVE_DIR}")

### 📥 Bước 3: Clone / Cập Nhật Mã Nguồn Từ Branch `new_framework`
Tự động lấy mã nguồn mới nhất chứa hàm `extract_cocoa_category_id` và bộ điều phối `scripts/run_ablation_cocoa.py`.

In [ ]:
%cd /content
BRANCH = "new_framework"

if not os.path.exists("/content/amodal_shape_prediction"):
    !git clone -b {BRANCH} https://github.com/vothenguyen/amodal_shape_prediction.git
else:
    %cd /content/amodal_shape_prediction
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd /content/amodal_shape_prediction

# Cài đặt các thư viện cần thiết
!pip install -q timm albumentations pycocotools tqdm pandas matplotlib
print("\n✅ Mã nguồn branch 'new_framework' và thư viện đã sẵn sàng 100%!")

### 🥥 Bước 4: Tự Động Nạp Dữ Liệu COCOA (Smart Search & NVMe SSD Extraction)
1. **Annotations (`annotations.zip`):** Tự động tìm trên Google Drive và giải nén siêu tốc vào SSD Colab.
2. **Images (`train2014`, `val2014`):** Quét Drive trước, nếu chưa có thì tự động tải trực tiếp từ `images.cocodataset.org` thẳng vào SSD Colab.

In [ ]:
import os
import sys
import glob
import shutil
import time

LOCAL_DATA_DIR = "/content/amodal_shape_prediction/data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
ANN_DIR = os.path.join(LOCAL_DATA_DIR, "annotations")
os.makedirs(ANN_DIR, exist_ok=True)

REQ_COCO_JSONS = [
    "COCO_amodal_train2014.json",
    "COCO_amodal_val2014.json"
]

SEARCH_DIRS_COCOA = [
    f"{DRIVE_DIR}/data/COCOA",
    f"{DRIVE_DIR}/data/cocoa",
    f"{DRIVE_DIR}/data",
    "/content/drive/MyDrive/data/COCOA",
    "/content/drive/MyDrive/data/cocoa",
    "/content/drive/MyDrive/COCOA",
    "/content/drive/MyDrive/cocoa",
    "/content/drive/MyDrive/CV+/data/COCOA",
    "/content/drive/MyDrive/amodal_shape_prediction/data/COCOA",
    "/content/drive/MyDrive/amodal_shape_prediction/data",
    f"{DRIVE_DIR}",
    "/content/drive/MyDrive",
    "/content/data/COCOA",
    "/content/data",
    "/content"
]

def find_file_in_search_dirs(patterns, search_dirs):
    for sdir in search_dirs:
        if not os.path.exists(sdir):
            continue
        for p in patterns:
            target = os.path.join(sdir, p)
            if os.path.isfile(target) or os.path.isdir(target):
                return target
            matches = glob.glob(os.path.join(sdir, p))
            if matches and (os.path.isfile(matches[0]) or os.path.isdir(matches[0])):
                return matches[0]
    return None

print("=" * 70)
print("🔍 THIẾT LẬP DỮ LIỆU COCOA")
print("=" * 70)

# 1. Xử lý Annotations
has_train_json = os.path.isfile(os.path.join(ANN_DIR, "COCO_amodal_train2014.json"))
has_val_json = os.path.isfile(os.path.join(ANN_DIR, "COCO_amodal_val2014.json"))

if has_train_json and has_val_json:
    print("✅ File annotations COCOA đã sẵn sàng tại data/annotations/")
else:
    print("🔎 Đang quét tìm annotations.zip trên Google Drive...")
    ann_archive = find_file_in_search_dirs(["annotations.zip", "*cocoa*annotation*.zip", "*annotations*.zip"], SEARCH_DIRS_COCOA)
    if ann_archive:
        print(f"📦 Tìm thấy annotations: {ann_archive}, đang giải nén...")
        !unzip -qo "{ann_archive}" -d "{LOCAL_DATA_DIR}"
    else:
        f_train = find_file_in_search_dirs(["COCO_amodal_train2014.json", "*amodal_train2014.json"], SEARCH_DIRS_COCOA)
        f_val = find_file_in_search_dirs(["COCO_amodal_val2014.json", "*amodal_val2014.json"], SEARCH_DIRS_COCOA)
        if f_train and f_val:
            shutil.copy(f_train, os.path.join(ANN_DIR, "COCO_amodal_train2014.json"))
            shutil.copy(f_val, os.path.join(ANN_DIR, "COCO_amodal_val2014.json"))
            print(f"✅ Đã sao chép các file JSON nhãn vào data/annotations/")

    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for req_f in REQ_COCO_JSONS:
            if req_f in files:
                t_path = os.path.join(ANN_DIR, req_f)
                s_path = os.path.join(root, req_f)
                if s_path != t_path and not os.path.exists(t_path):
                    shutil.copy(s_path, t_path)

# 2. Xử lý Ảnh
train_img_dir = os.path.join(LOCAL_DATA_DIR, "train2014")
val_img_dir = os.path.join(LOCAL_DATA_DIR, "val2014")
has_train_imgs = os.path.isdir(train_img_dir) and len(os.listdir(train_img_dir)) >= 10000
has_val_imgs = os.path.isdir(val_img_dir) and len(os.listdir(val_img_dir)) >= 5000

if not has_train_imgs:
    train_drive = find_file_in_search_dirs(["train2014", "train2014.zip"], SEARCH_DIRS_COCOA)
    if train_drive and os.path.isdir(train_drive):
        !rm -rf "{train_img_dir}" && ln -sf "{train_drive}" "{train_img_dir}"
    elif train_drive and train_drive.endswith(".zip"):
        !unzip -q "{train_drive}" -d "{LOCAL_DATA_DIR}"
    else:
        print("🌐 Tải ảnh train2014 từ images.cocodataset.org thẳng vào SSD Colab...")
        %cd "{LOCAL_DATA_DIR}"
        !wget -q http://images.cocodataset.org/zips/train2014.zip && unzip -q train2014.zip && rm train2014.zip
        %cd /content/amodal_shape_prediction

if not has_val_imgs:
    val_drive = find_file_in_search_dirs(["val2014", "val2014.zip"], SEARCH_DIRS_COCOA)
    if val_drive and os.path.isdir(val_drive):
        !rm -rf "{val_img_dir}" && ln -sf "{val_drive}" "{val_img_dir}"
    elif val_drive and val_drive.endswith(".zip"):
        !unzip -q "{val_drive}" -d "{LOCAL_DATA_DIR}"
    else:
        print("🌐 Tải ảnh val2014 từ images.cocodataset.org thẳng vào SSD Colab...")
        %cd "{LOCAL_DATA_DIR}"
        !wget -q http://images.cocodataset.org/zips/val2014.zip && unzip -q val2014.zip && rm val2014.zip
        %cd /content/amodal_shape_prediction

print("\n🎉 DỮ LIỆU COCOA ĐÃ SẴN SÀNG HUẤN LUYỆN!")
print(f"   • Train JSON: {os.path.exists(os.path.join(ANN_DIR, 'COCO_amodal_train2014.json'))}")
print(f"   • Val JSON  : {os.path.exists(os.path.join(ANN_DIR, 'COCO_amodal_val2014.json'))}")
print(f"   • Train Imgs: {len(os.listdir(train_img_dir)) if os.path.isdir(train_img_dir) else 0:,} ảnh")
print(f"   • Val Imgs  : {len(os.listdir(val_img_dir)) if os.path.isdir(val_img_dir) else 0:,} ảnh")

### 🧪 Bước 5 (Tùy chọn): Smoke Test Nhanh (1 Epoch trên 50 Mẫu)
Chạy thử nhanh Row 4 và Row 7 trên 50 mẫu để kiểm tra toàn vẹn luồng nạp dữ liệu, forward pass, backward pass và tính metrics trong ~30 giây.

In [ ]:
!python scripts/run_ablation_cocoa.py \
    --rows 4 7 \
    --mode all \
    --epochs 1 \
    --subset-size 50 \
    --val-subset-size 50 \
    --batch-size 16 \
    --num-workers 4 \
    --amp \
    --tag {EXP_TAG} \
    --drive-dir {DRIVE_DIR}

---
# 🏆 BƯỚC 6: CHỌN CHIẾN LƯỢC HUẤN LUYỆN CHÍNH THỨC (TABLE V)

### 💡 Gợi ý lựa chọn:
- **LỰA CHỌN 1 (KHUYÊN DÙNG - TIẾT KIỆM THỜI GIAN ~1.5 GIỜ):**  
  Chạy duy nhất 3 cấu hình bị ảnh hưởng bởi lỗi Category ID: **Row 4 (Main Config), Row 6 (Full Config), Row 7 (Proposed SG-GBA)**.  
  *(Row 1, 2, 3, 5 không có Category Embedding nên không cần phí thêm thời gian chạy lại).*

- **LỰA CHỌN 2 (CHUẨN MỰC HỌC THUẬT ĐẦY ĐỦ ~3 GIỜ):**  
  Chạy mới toanh tuần tự từ **Row 1 đến Row 7** trên cùng một phiên bản git để bảng Table V hoàn toàn đồng nhất từ A đến Z.

### 🌟 LỰA CHỌN 1: Huấn Luyện & Đánh Giá 3 Cấu Hình Có Category Embedding (Row 4, 6, 7)

In [ ]:
# Huấn luyện và đánh giá Row 4, Row 6, Row 7 (30 Epochs mỗi row)
!python scripts/run_ablation_cocoa.py \
    --rows 4 6 7 \
    --mode all \
    --epochs 30 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 8 \
    --lr 1e-4 \
    --amp \
    --tag {EXP_TAG} \
    --drive-dir {DRIVE_DIR}

### 🌟 LỰA CHỌN 2: Huấn Luyện & Đánh Giá TOÀN BỘ 7 CẤU HÌNH (Row 1 -> Row 7)

In [ ]:
# Huấn luyện và đánh giá toàn bộ từ Row 1 đến Row 7 (30 Epochs mỗi row)
# Bỏ comment dòng bên dưới nếu bạn muốn chạy trọn vẹn cả 7 cấu hình:
# !python scripts/run_ablation_cocoa.py \
#     --rows 1 2 3 4 5 6 7 \
#     --mode all \
#     --epochs 30 \
#     --batch-size 16 \
#     --accumulation-steps 1 \
#     --num-workers 8 \
#     --lr 1e-4 \
#     --amp \
#     --tag {EXP_TAG} \
#     --drive-dir {DRIVE_DIR}

---
# 🎯 CÁC Ô CHẠY LẺ TỪNG CẤU HÌNH ĐỘC LẬP (NẾU CẦN CHẠY RIÊNG TỪNG ROW)

#### 🔹 Row 1: 4ch (RGB+Vis) | No Edge | No Emb | No Spatial | BCE Loss

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 1 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

#### 🔹 Row 2: 5ch (RGB+Vis+Edge) | Edge | No Emb | No Spatial | BCE Loss

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 2 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

#### 🔹 Row 3: 5ch (RGB+Vis+Edge) | Edge | No Emb | No Spatial | Occ-Aware Loss (5x)

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 3 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

#### 🔹 Row 4 (Main Config): 5ch | Edge | Category Embedding (91) | No Spatial | Occ-Aware Loss (5x)

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 4 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

#### 🔹 Row 5: 5ch | Edge | No Emb | Spatial Attention 7x7 | Occ-Aware Loss (5x)

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 5 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

#### 🔹 Row 6 (Full Config): 5ch | Edge | Category Embedding (91) | Spatial Attention 7x7 | Occ-Aware Loss (5x)

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 6 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

#### 🔹 Row 7 (Proposed SG-GBA): 5ch | Edge | Category Embedding (91) | SGGatedBottleneck (FiLM + Spatial 7x7 + Gate Alpha) | Occ-Aware Loss (5x)

In [ ]:
!python scripts/run_ablation_cocoa.py --rows 7 --mode all --epochs 30 --batch-size 16 --num-workers 8 --amp --tag {EXP_TAG} --drive-dir {DRIVE_DIR}

---
# 📊 BƯỚC 7: BÁO CÁO TỔNG HỢP TABLE V & TRỰC QUAN HÓA KẾT QUẢ

Đọc các file đánh giá JSON từ thư mục `{DRIVE_DIR}/results/cocoa_ablation_v2/`, hiển thị Pandas DataFrame và vẽ biểu đồ so sánh.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

summary_file = f"{RESULTS_SAVE_DIR}/table_v_cocoa_summary.json"
md_file = f"{RESULTS_SAVE_DIR}/table_v_cocoa_summary.md"

print("=" * 80)
print("🏆 BÁO CÁO KẾT QUẢ ABLATION STUDY COCOA (TABLE V) - THƯ MỤC V2")
print("=" * 80)

if os.path.exists(summary_file):
    with open(summary_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    df = pd.DataFrame(records)
    df = df.rename(columns={
        "row": "Row",
        "name": "Cấu hình",
        "mIoU": "Overall mIoU (%)",
        "inv_mIoU": "Invisible mIoU (%)",
        "dice": "Dice (%)",
        "precision": "Precision (%)",
        "recall": "Recall (%)",
        "gate_alpha": "Gate Alpha"
    })
    display(df)

    # Vẽ biểu đồ so sánh cột
    plt.figure(figsize=(10, 5), dpi=150)
    x = range(len(df))
    width = 0.35
    plt.bar([i - width/2 for i in x], df["Overall mIoU (%)"], width=width, label="Overall mIoU (%)", color="#2b5c8f")
    plt.bar([i + width/2 for i in x], df["Invisible mIoU (%)"], width=width, label="Invisible mIoU (%)", color="#e26d5c")
    plt.xlabel("Cấu hình")
    plt.ylabel("Độ đo IoU (%)")
    plt.title("So Sánh Hiệu Năng Ablation Study Trên Tập COCOA (Table V)")
    plt.xticks(x, [f"Row {r}" for r in df["Row"]])
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ Chưa tìm thấy file {summary_file}. Đảm bảo bạn đã hoàn thành bước huấn luyện & đánh giá ít nhất 1 cấu hình.")

print("\n📂 Danh sách checkpoints lưu trữ an toàn trên Google Drive:")
!ls -lh "{CKPT_SAVE_DIR}"